# KNN Analysis - Lab 02 - EIF420O Artificial Intelligence
## Universidad Nacional de Costa Rica
### Dataset: BankChurners (Bank Customer Churn Prediction)
**Target:** Attrition_Flag (Existing Customer / Attrited Customer)

This notebook implements the KNN classification analysis using the Supervisado class pattern from the course package.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

## 1. Supervisado Class (Course Package)

In [ ]:
class Supervisado:
    def __init__(self, df):
        self.__df = df

    @property
    def df(self):
        return self.__df

    @df.setter
    def df(self, p_df):
        self.__df = p_df

    def preparar_datos(self, target_col='target', test_size=0.25, random_state=42):
        X = self.__df.drop(columns=[target_col])
        X = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
        y = self.__df[target_col]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y)
        return X_train, X_test, y_train, y_test, y

    def modeloKNN(self, X_train, y_train, n_neighbors=5, algorithm='auto',
                  weights='uniform', metric='minkowski', p=2):
        model = KNeighborsClassifier(n_neighbors=n_neighbors, algorithm=algorithm,
                                     weights=weights, metric=metric, p=p)
        model.fit(X_train, y_train)
        return model

    def predecir(self, model, X_test):
        return model.predict(X_test)

    def evaluar(self, y_test, y_pred, y):
        MC = confusion_matrix(y_test, y_pred)
        return self.indices_general(MC, list(np.unique(y)))

    def indices_general(self, MC, nombres=None):
        precision_global = np.sum(MC.diagonal()) / np.sum(MC)
        error_global = 1 - precision_global
        precision_categoria = pd.DataFrame(MC.diagonal() / np.sum(MC, axis=1)).T
        if nombres is not None:
            precision_categoria.columns = nombres
        return {"Matriz de Confusión": MC, "Precisión Global": precision_global,
                "Error Global": error_global, "Precisión por categoría": precision_categoria}

## 2. Data Loading and Preparation

In [ ]:
# Load dataset
df = pd.read_csv('BankChurners.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
# Prepare data
df = df.drop(columns=['ID'])
le_target = LabelEncoder()
df['target'] = le_target.fit_transform(df['Attrition_Flag'])
df = df.drop(columns=['Attrition_Flag'])

cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

print("Target classes:", le_target.classes_)
print("Target distribution:")
print(df['target'].value_counts())
print("\nShape:", df.shape)
df.head()

## 3. Standard KNN (k=5, uniform, euclidean)

In [ ]:
sup = Supervisado(df)
X_train, X_test, y_train, y_test, y = sup.preparar_datos(target_col='target', random_state=42)

# Standard model
model_std = sup.modeloKNN(X_train, y_train, n_neighbors=5, algorithm='auto',
                          weights='uniform', metric='minkowski', p=2)
y_pred_std = sup.predecir(model_std, X_test)
indices_std = sup.evaluar(y_test, y_pred_std, y)

print("Global Precision:", f"{indices_std['Precisión Global']:.4f}")
print("Global Error:", f"{indices_std['Error Global']:.4f}")
print("Precision by category:")
print(indices_std['Precisión por categoría'])
print("\nConfusion Matrix:")
print(indices_std['Matriz de Confusión'])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(confusion_matrix=indices_std['Matriz de Confusión'],
                       display_labels=le_target.classes_).plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix - Standard KNN (k=5, uniform, euclidean)')
plt.tight_layout()
plt.show()

## 4. KNN Variations (Hyperparameter Exploration)

In [ ]:
# Systematic exploration
resultados = []
k_values = list(range(1, 21))
algorithms = ['auto', 'ball_tree', 'kd_tree', 'brute']
weight_opts = ['uniform', 'distance']
metrics = [('minkowski', 2, 'Euclidean'), ('minkowski', 1, 'Manhattan'), ('minkowski', 3, 'Minkowski(p=3)')]

for k in k_values:
    for algo in algorithms:
        for w in weight_opts:
            for metric, p, metric_name in metrics:
                model = sup.modeloKNN(X_train, y_train, n_neighbors=k,
                                      algorithm=algo, weights=w, metric=metric, p=p)
                y_pred = sup.predecir(model, X_test)
                indices = sup.evaluar(y_test, y_pred, y)
                resultados.append({'k': k, 'algorithm': algo, 'weights': w,
                                   'metric': metric_name,
                                   'precision_global': indices['Precisión Global'],
                                   'error_global': indices['Error Global'],
                                   'y_pred': y_pred})

df_results = pd.DataFrame([{k: v for k, v in r.items() if k != 'y_pred'} for r in resultados])
print(f"Total configurations: {len(df_results)}")
print(f"Best precision: {df_results['precision_global'].max():.4f}")
print(f"Worst precision: {df_results['precision_global'].min():.4f}")

## 5. Visualizations

In [ ]:
# Precision vs k
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, w in enumerate(['uniform', 'distance']):
    ax = axes[i]
    subset = df_results[(df_results['weights'] == w) & (df_results['algorithm'] == 'auto')]
    for m in subset['metric'].unique():
        data = subset[subset['metric'] == m]
        ax.plot(data['k'], data['precision_global'], marker='o', markersize=3, label=m)
    ax.set_xlabel('k'); ax.set_ylabel('Global Precision')
    ax.set_title(f'Weights: {w}'); ax.legend(); ax.grid(alpha=0.3)
    ax.set_xticks(range(1, 21))
fig.suptitle('KNN: Global Precision vs Number of Neighbors', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Heatmap
subset = df_results[(df_results['weights'] == 'distance') & (df_results['algorithm'] == 'auto')]
pivot = subset.pivot_table(index='metric', columns='k', values='precision_global')
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax, linewidths=0.5)
ax.set_title('KNN Precision Heatmap (weights=distance, algorithm=auto)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Top 10
top10 = df_results.nlargest(10, 'precision_global').reset_index(drop=True)
top10['label'] = top10.apply(lambda r: f"k={int(r['k'])}, {r['weights']}, {r['metric']}", axis=1)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(range(len(top10)), top10['precision_global'],
               color=sns.color_palette('viridis', len(top10)), edgecolor='black')
ax.set_yticks(range(len(top10))); ax.set_yticklabels(top10['label'])
ax.set_xlabel('Global Precision'); ax.set_title('Top 10 KNN Configurations', fontweight='bold')
for bar, val in zip(bars, top10['precision_global']):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
top10[['k','algorithm','weights','metric','precision_global']]

## 6. Best Model Analysis

In [ ]:
# Best model
idx_best = df_results['precision_global'].idxmax()
best = resultados[idx_best]
best_indices = sup.evaluar(y_test, best['y_pred'], y)

print("Best Configuration:")
print(f"  k={best['k']}, algorithm={best['algorithm']}, weights={best['weights']}, metric={best['metric']}")
print(f"\nGlobal Precision: {best_indices['Precisión Global']:.4f}")
print(f"Global Error: {best_indices['Error Global']:.4f}")
print(f"\nPrecision by category:")
print(best_indices['Precisión por categoría'])
print(f"\nConfusion Matrix:")
print(best_indices['Matriz de Confusión'])
print(f"\nClassification Report:")
print(classification_report(y_test, best['y_pred'], target_names=le_target.classes_))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(confusion_matrix=indices_std['Matriz de Confusión'],
                       display_labels=le_target.classes_).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Standard KNN (k=5)')
ConfusionMatrixDisplay(confusion_matrix=best_indices['Matriz de Confusión'],
                       display_labels=le_target.classes_).plot(ax=axes[1], cmap='Greens')
axes[1].set_title(f"Best KNN (k={best['k']}, {best['weights']}, {best['metric']})")
plt.tight_layout(); plt.show()